# Load Cushion Telemetry from GitHub

Downloads the Kalogon smart-cushion sensor telemetry dataset from the
[GitHub Release](https://github.com/KalogonTech/kalogon_sci_hackathon_2026/releases/tag/v1.0.0)
and registers it as a Delta table in Unity Catalog.

**Dataset**: 9.4 M rows, 23 columns, 113 sessions, 10 Hz sample rate.

In [ ]:
DEFAULT_CATALOG = "main"
DEFAULT_SCHEMA  = "kalogon"

try:
    dbutils.widgets.text("catalog", DEFAULT_CATALOG, "Catalog")
    dbutils.widgets.text("schema",  DEFAULT_SCHEMA,  "Schema")
    catalog = dbutils.widgets.get("catalog")
    schema  = dbutils.widgets.get("schema")
except Exception:
    catalog = DEFAULT_CATALOG
    schema  = DEFAULT_SCHEMA

spark.sql(f"USE CATALOG {catalog}")
print(f"Catalog: {catalog}  |  Schema: {schema}")

## Create schema and Volume

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.raw")
print(f"Volume ready: /Volumes/{catalog}/{schema}/raw")

## Download Parquet files from GitHub Release

Uses the GitHub API to discover all `.parquet` assets from the `v1.0.0`
release, then streams them into the UC Volume.

In [ ]:
import requests
import os
import time

REPO  = "KalogonTech/kalogon_sci_hackathon_2026"
TAG   = "v1.0.0"
DEST  = f"/Volumes/{catalog}/{schema}/raw/cushion_telemetry"

os.makedirs(DEST, exist_ok=True)

existing = set(os.listdir(DEST))
if any(f.endswith(".parquet") for f in existing):
    print(f"Parquet files already present in {DEST} — skipping download.")
    print(f"  Delete the directory to force re-download.")
else:
    api_url = f"https://api.github.com/repos/{REPO}/releases/tags/{TAG}"
    release = requests.get(api_url, timeout=30).json()
    assets  = [a for a in release.get("assets", []) if a["name"].endswith(".parquet")]
    print(f"Found {len(assets)} parquet assets in release {TAG}")

    for i, asset in enumerate(assets):
        name = asset["name"]
        url  = asset["browser_download_url"]
        dest_path = os.path.join(DEST, name)
        size_mb   = asset["size"] / (1024 * 1024)

        print(f"  [{i+1}/{len(assets)}] Downloading {name} ({size_mb:.1f} MB)...", end=" ")
        t0 = time.time()
        resp = requests.get(url, stream=True, timeout=300)
        resp.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
        elapsed = time.time() - t0
        print(f"done ({elapsed:.1f}s)")

    print(f"\nAll files saved to {DEST}")

## Create Delta table from Parquet

In [ ]:
fq_table = f"{catalog}.{schema}.cushion_telemetry"

df = spark.read.parquet(DEST)
df.write.mode("overwrite").saveAsTable(fq_table)

row_count = spark.table(fq_table).count()
print(f"Created table {fq_table} with {row_count:,} rows")

## Verify

In [ ]:
display(spark.table(fq_table).limit(20))

In [ ]:
display(
    spark.sql(f"""
        SELECT
            count(*)           AS total_rows,
            count(DISTINCT session_id) AS sessions,
            min(timestamp)     AS earliest,
            max(timestamp)     AS latest
        FROM {fq_table}
    """)
)